In [ ]:
import os

data_dir = os.path.abspath("data/public")
project_dir = os.path.abspath("data/project")
chrombpnet_dir = os.path.join(project_dir, "chrombpnet")

In [ ]:
from chrombpnet.data import DefaultDataFile, get_default_data_path
import chrombpnet.helpers.preprocessing.auto_shift_detect as auto_shift_detect
import chrombpnet.training.utils.one_hot as one_hot
import pysam
import random
import pandas as pd
from collections import Counter
import subprocess
import os

### Preprocessing

In [ ]:
# Ingest metadata
beta_train = pd.read_csv(f"{chrombpnet_dir}/beta_train_metadata.csv", usecols = ["Cell","disease_harmonized"])
beta_holdout = pd.read_csv(f"{chrombpnet_dir}/beta_heldout_metadata.csv", usecols = ["Cell","disease_harmonized"])
beta_metadata = pd.concat((beta_train, beta_holdout))
alpha_train = pd.read_csv(f"{chrombpnet_dir}/alpha_train_metadata.csv", usecols = ["Cell","disease_harmonized"])
alpha_holdout = pd.read_csv(f"{chrombpnet_dir}/alpha_heldout_metadata.csv", usecols = ["Cell","disease_harmonized"])
alpha_metadata = pd.concat((alpha_train, alpha_holdout))
metadata_dict = {'beta': beta_metadata, 'alpha': alpha_metadata}

In [ ]:
## Sample reads from tabix file
# Parameters
celltypes = ["alpha", "beta"]
states = ["ND", "T2D"]
chrom_size_file = f"{chrombpnet_dir}/data/hg38.chrom.sizes"
genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
n_sample = 10000

# Define the path for the fragment file
for celltype in celltypes:
    for state in states:
        fragment_file = f"{data_dir}/scATACseq/ArchR/IsletProject/GroupFragments/fragments/" + str(celltype) + ".fragments.tsv.gz"
        output_bedgraph = f"{chrombpnet_dir}/data/{celltype}_{state}_coverage.bedGraph"
        barcode_set = set(list(metadata_dict[celltype][metadata_dict[celltype]['disease_harmonized'] == state]['Cell']))
        
        # Read the chromosome sizes
        chr_size = pd.read_csv(chrom_size_file, sep="\t", header=None)
        allowed_chroms = [f"chr{i}" for i in range(1, 23)] + ["chrX", "chrY"]
        chr_size = chr_size[chr_size[0].isin(allowed_chroms)].reset_index(drop=True)
        
        # Open the fragment file
        with pysam.TabixFile(fragment_file, 'r') as tb:    
            # Sample using randomized regions
            sampled_reads = []
            counter = 0
            while counter <= n_sample:
                random_row = chr_size.sample(n=1)
                rand_num = random.randint(0, random_row.iloc[0,1]-1000)
                records = list(tb.fetch(random_row.iloc[0,0], rand_num, rand_num+1000))
                if records:
                    read = random.choice(records).strip().split('\t')
                    if counter == 0:
                        sampled_reads = pd.DataFrame([read], index = [counter])
                    else:
                        sampled_reads = pd.concat((sampled_reads, pd.DataFrame([read], index = [counter])))
                    counter += 1
            
            # Postprocessing
            sampled_reads[3] = 1000
            sampled_reads[4] = 0
            sampled_reads[5] = '+'
            sampled_reads = sampled_reads.rename({0:'chr', 1:'start', 2:'end', 3:'x1', 4:'x2', 5:'strand'}, axis=1)
            
            # Finalize
            plus_reads = sampled_reads.copy()
            plus_reads = plus_reads[plus_reads['strand']=="+"].iloc[:,:3]
            minus_reads = sampled_reads.copy()
            minus_reads['strand'] = "-"
            minus_reads = minus_reads[minus_reads['strand']=="-"].iloc[:,:3]
            
            # Calculate shifts
            plus_pwm, minus_pwm = auto_shift_detect.get_pwms(plus_reads, minus_reads, genome_fasta_path)
            ref_motifs_file=get_default_data_path(DefaultDataFile.atac_ref_motifs)
            ref_plus_pwms, ref_minus_pwms = auto_shift_detect.get_ref_pwms(ref_motifs_file)
            plus_shift, minus_shift = auto_shift_detect.compute_shift_ATAC(ref_plus_pwms, ref_minus_pwms, plus_pwm, minus_pwm)
            plus_shift_delta, minus_shift_delta = 4-plus_shift, -4-minus_shift
            
            # Summarize coverage
            coverage = Counter()
            for chrom in chr_size[0]:
                records = tb.fetch(chrom)
                for record in records:
                    fields = record.strip().split('\t')
                    if fields[3] in barcode_set:
                        chrom = fields[0]
                        start = int(fields[1])
                        end = int(fields[2])
                        coverage[(chrom, max(0, start + plus_shift_delta))] += 1
                        coverage[(chrom, max(0, end + minus_shift_delta - 1))] += 1
                        last_pos_chrom = chrom
                        last_pos_start = max(0, start + plus_shift_delta)
        
        # Setup chromosomes
        chrom_sizes = {}
        for idx in chr_size.index:
            chrom_sizes[chr_size.iloc[idx][0]] = int(chr_size.iloc[idx][1])
        
        # Process
        with open(output_bedgraph, 'w') as out:
            for chrom in chrom_sizes:
                # Extract coverage for this chrom, sorted by position
                positions = sorted(pos for (c, pos) in coverage if c == chrom)
                if not positions:
                    continue
                
                start = positions[0]
                prev_cov = coverage[(chrom, start)]
                end = start + 1  # bedGraph intervals are zero-based half-open: [start, end)
                
                for pos in positions[1:]:
                    cov = coverage[(chrom, pos)]
                    # Extend the current interval if coverage is same and positions are continuous
                    if cov == prev_cov and pos == end:
                        end += 1
                    else:
                        # Write the previous interval if coverage > 0
                        if prev_cov > 0:
                            out.write(f"{chrom}\t{start}\t{end}\t{prev_cov}\n")
                        start = pos
                        end = pos + 1
                        prev_cov = cov
                
                # Write last interval
                if prev_cov > 0:
                    out.write(f"{chrom}\t{start}\t{end}\t{prev_cov}\n")

In [ ]:
# Sort and make bigWig
celltypes = ["alpha", "beta"]
chrom_size_file = f"{chrombpnet_dir}/data/hg38.chrom.sizes"
states = ["ND", "T2D"]
for celltype in celltypes:
    for state in states:
        output_bedgraph = f"{chrombpnet_dir}/data/{celltype}_{state}_coverage.bedGraph"
        sorted_bedgraph = f"{chrombpnet_dir}/data/{celltype}_{state}_coverage_sorted.bedGraph"
        output_bigWig = f"{chrombpnet_dir}/data/{celltype}_{state}_coverage_unstranded.bw"
        
        # Sort the bedGraph
        command = ['sort', '-k1,1', '-k2,2n', output_bedgraph]
        with open(sorted_bedgraph, 'w') as outfile:
            subprocess.run(command, stdout=outfile, check=True)
        
        # Make a bigWig
        subprocess.run(["bedGraphToBigWig", sorted_bedgraph, chrom_size_file, output_bigWig])

### Quality control the bigWigs

In [ ]:
import chrombpnet.helpers.preprocessing.analysis.build_pwm_from_bigwig as build_pwm_from_bigwig
import json
import argparse

In [ ]:
celltypes = ["alpha", "beta"]
states = ["ND", "T2D"]
genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
chrom_size_file = f"{chrombpnet_dir}/data/hg38.chrom.sizes"
for celltype in celltypes:
    for state in states:
        for fold in range(5):
            # Make output directory
            output_directory = f"{chrombpnet_dir}/data/fold" + str(fold) + "/qc/"
            if not os.path.exists(output_directory):
                os.makedirs(output_directory)
                
            # Load the fold
            fold_path = f"{chrombpnet_dir}/data/fold_" + str(fold) + ".json"
            folds = json.load(open(fold_path))
    
            # Set the arguments
            args = argparse.Namespace(
                bigwig=f"{chrombpnet_dir}/data/{celltype}_{state}_coverage_unstranded.bw",
                genome=genome_fasta_path,
                pwm_width=24,
                chrom_sizes = chrom_size_file,
                output_prefix = f"{output_directory}{celltype}_{state}_bigWig_qc",
                chr = folds["valid"][0]
            )
    
            # Call function
            build_pwm_from_bigwig.main(args)

## Adjust bias models

In [ ]:
import chrombpnet.helpers.hyperparameters.find_chrombpnet_hyperparams as find_chrombpnet_hyperparams
import os
import argparse
from tqdm.contrib.concurrent import process_map
from functools import partial

In [ ]:
# Source https://github.com/kundajelab/chrombpnet/blob/master/chrombpnet/helpers/hyperparameters/find_chrombpnet_hyperparams.py
# Seperate CPU and GPU tasks for better parallel processing

import argparse
import pyfaidx
import pyBigWig
import pandas as pd
import numpy as np
import os
from tensorflow import keras
import json
import numpy as np
import chrombpnet.training.utils.one_hot as one_hot
import tensorflow as tf
from tensorflow.keras.utils import get_custom_objects
from tensorflow.keras.models import load_model
import chrombpnet.training.utils.losses as losses

def filter_edge_regions(peaks_df, bw, width,):
    """
    Filter regions in bed file that are on edges i.e regions that cannot be used to construct
    input length + jitter length of the sequence
    """
    input_shape = peaks_df.shape[0]

    # left edge case
    filtered = np.array((peaks_df['start'] + peaks_df['summit'] - width//2) < 0)
    peaks_df = peaks_df[~filtered]
    num_filtered = sum(filtered)

    # right edge case
    chrom_to_sizes = bw.chroms()
    filtered = []
    for i, r in peaks_df.iterrows():
        if r['start'] + r['summit'] + width//2 > chrom_to_sizes.get(r['chr'], 0) :
            filtered.append(True)
        else:
            filtered.append(False)
    filtered=np.array(filtered)
    peaks_df = peaks_df[~filtered]
    num_filtered += sum(filtered)
    
    return peaks_df

def get_seqs_cts(genome, bw, peaks_df, input_width=2114, output_width=1000):
    """
    Output counts (not log counts)
    Output one-hot encoded sequence
    """
    vals = []
    seqs = []
    for i, r in peaks_df.iterrows():
        sequence = str(genome[r['chr']][(r['start']+r['summit'] - input_width//2):(r['start'] + r['summit'] + input_width//2)])
        seqs.append(sequence)
        bigwig_vals=np.nan_to_num(bw.values(r['chr'], 
                            (r['start'] + r['summit']) - output_width//2,
                            (r['start'] + r['summit']) + output_width//2))
        vals.append(bigwig_vals)
    return (np.sum(np.array(vals),axis=1), one_hot.dna_to_one_hot(seqs))

def load_model_wrapper(model_h5):
    # read .h5 model
    custom_objects={"tf": tf, "multinomial_nll":losses.multinomial_nll}    
    get_custom_objects().update(custom_objects)    
    model=load_model(model_h5)
    print("got the model")
    model.summary()
    return model
    

def parse_data_args():
    parser=argparse.ArgumentParser(description="find hyper-parameters for chrombpnet defined in src/training/models/chrombpnet_with_bias_model.py")
    parser.add_argument("-g", "--genome", type=str, required=True, help="Genome fasta")
    parser.add_argument("-i", "--bigwig", type=str, required=True, help="Bigwig of tn5 insertions. Ensure it is +4/-4 shifted")
    parser.add_argument("-p", "--peaks", type=str, required=True, help="10 column bed file of peaks. Sequences and labels will be extracted centered at start (2nd col) + summit (10th col).")
    parser.add_argument("-n", "--nonpeaks", type=str, required=True, help="10 column bed file of non-peak regions, centered at summit (10th column)")
    parser.add_argument("-sr", "--negative_sampling_ratio", type=float, default=0.1, help="Ratio of negatives to positive samples per epoch")
    parser.add_argument("-oth", "--outlier-threshold", type=float, default=0.9999, help="threshold to use to filter outlies")
    parser.add_argument("-j", "--max-jitter", type=int, required=True, default=500, help="Maximum jitter applied on either side of region (default 500 for chrombpnet model)")
    parser.add_argument("-fl", "--chr-fold-path", type=str, required=True, help="Fold information - dictionary with test,valid and train keys and values with corresponding chromosomes")
    return parser

def parse_model_args(parser):
    # arguments here defined the following model - src/training/models/chrombpnet_with_bias_model.py
    parser.add_argument("-il", "--inputlen", type=int, required=True, help="Sequence input length")
    parser.add_argument("-ol", "--outputlen", type=int, required=True, help="Prediction output length")
    parser.add_argument("-fil", "--filters", type=int, default=512, help="Number of filters to use in chrombpnet mode")
    parser.add_argument("-dil", "--n-dilation-layers", type=int, default=8, help="Number of dilation layers to use in chrombpnet model")
    parser.add_argument("-b", "--bias-model-path", type=str, required=True, help="path of bias model")
    parser.add_argument("-op", "--output-prefix", help="output prefix for storing hyper-param TSV for chrombpnet")
    args = parser.parse_args()
    return args

def adjust_bias_model_logcounts(bias_model, seqs, cts):
    """
    Given a bias model, sequences and associated counts, the function adds a 
    constant to the output of the bias_model's logcounts that minimises squared
    error between predicted logcounts and observed logcounts (infered from 
    cts). This simply reduces to adding the average difference between observed 
    and predicted to the "bias" (constant additive term) of the Dense layer.
    Typically the seqs and counts would correspond to training nonpeak regions.
    ASSUMES model_bias's last layer is a dense layer that outputs logcounts. 
    This would change if you change the model.
    """

    # safeguards to prevent misuse
    #assert(bias_model.layers[-1].name == "logcount_predictions")
    assert(bias_model.layers[-1].name == "logcounts" or bias_model.layers[-1].name == "logcount_predictions")
    assert(bias_model.layers[-1].output_shape==(None,1))
    assert(isinstance(bias_model.layers[-1], keras.layers.Dense))

    print("Predicting within adjust counts")
    _, pred_logcts = bias_model.predict(seqs, verbose=True)
    delta = np.mean(np.log(1+cts) - pred_logcts.ravel())

    dw, db = bias_model.layers[-1].get_weights()
    bias_model.layers[-1].set_weights([dw, db+delta])
    return bias_model


def get_hyper_params(args): 

    # read the fold information - we will evaluate hyperparams on the train+valid set and do nothing on the test set 
    splits_dict=json.load(open(args.chr_fold_path))
    chroms_to_keep=splits_dict["train"]+splits_dict["valid"]
    test_chroms_to_keep=splits_dict["test"]

    # read from bigwigw and fasta file
    bw = pyBigWig.open(args.bigwig) 
    genome = pyfaidx.Fasta(args.genome)

    # read peaks and non peaks    
    in_peaks =  pd.read_csv(args.peaks,
                           sep='\t',
                           header=None,
                           names=["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"])
    in_nonpeaks =  pd.read_csv(args.nonpeaks,
                           sep='\t',
                           header=None,
                           names=["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"])

    assert(in_peaks.shape[0] != 0) # peaks file is empty
    assert(in_nonpeaks.shape[0] !=0) # non peaks file is empty
    assert(args.inputlen >= args.outputlen) # inputlen should be greater than the outputlen 
                                            # inputlen and outlen are chosen based on the filters and dilations layers used

    # get train/valid peaks and test peaks seperately
    peaks = in_peaks[(in_peaks["chr"].isin(chroms_to_keep))]
    test_peaks = in_peaks[(in_peaks["chr"].isin(test_chroms_to_keep))]

    nonpeaks = in_nonpeaks[(in_nonpeaks["chr"].isin(chroms_to_keep))]
    test_nonpeaks = in_nonpeaks[(in_nonpeaks["chr"].isin(test_chroms_to_keep))]

    # step 1 filtering: filter peaks that are in the edges - which prevents us from making the inputlen regions - do this for all splits
    peaks = filter_edge_regions(peaks, bw, args.inputlen+2*args.max_jitter)
    test_peaks = filter_edge_regions(test_peaks, bw, args.inputlen)
   
    nonpeaks = filter_edge_regions(nonpeaks, bw, args.inputlen)
    test_nonpeaks = filter_edge_regions(test_nonpeaks, bw, args.inputlen)

    # step 2 filtering: filter peaks that are outliers in train and valid set - no filtering on test set
    peak_cnts, _ = get_seqs_cts(genome, bw, peaks, args.inputlen, args.outputlen)
    nonpeak_cnts, nonpeak_seqs = get_seqs_cts(genome, bw, nonpeaks, args.inputlen, args.outputlen)    
    assert(len(peak_cnts) == peaks.shape[0])
    assert(len(nonpeak_cnts) == nonpeaks.shape[0])


    if args.negative_sampling_ratio > 0:
        final_cnts = np.concatenate((peak_cnts,np.random.choice(nonpeak_cnts, replace=False, size=(int(args.negative_sampling_ratio*len(peak_cnts))))))
    else:
        final_cnts = peak_cnts

    upper_thresh = np.quantile(final_cnts, args.outlier_threshold)
    lower_thresh = np.quantile(final_cnts, 1-args.outlier_threshold)

    peaks = peaks[(peak_cnts< upper_thresh) & (peak_cnts>lower_thresh)]
    nonpeaks = nonpeaks[(nonpeak_cnts< upper_thresh) & (nonpeak_cnts>lower_thresh)]

    # combine train valid and test peak set and store them in a new file
    if nonpeaks.shape[0] > peaks.shape[0]:
        train_nonpeaks = nonpeaks.sample(n=peaks.shape[0], random_state=1)
    else:
        train_nonpeaks = nonpeaks
        
    frames = [peaks, test_peaks]
    all_peaks = pd.concat(frames)
    all_peaks.to_csv("{}filtered.peaks.bed".format(args.output_prefix), sep="\t",  header=False, index=False)
    frames = [train_nonpeaks, test_nonpeaks]
    all_nonpeaks = pd.concat(frames)
    all_nonpeaks.to_csv("{}filtered.nonpeaks.bed".format(args.output_prefix), sep="\t", header=False, index=False)

    # find counts loss weight for model training - using train and validation set
    counts_loss_weight = np.median(final_cnts[(final_cnts <= upper_thresh) & (final_cnts>=lower_thresh)])/10
    assert(counts_loss_weight != 0)
    #assert(counts_loss_weight > 1.0) # counts loss weight can go less than 1.0 if you have very low-read depth - make sure you have enough density in counts
                                     # check peak-calling

    if counts_loss_weight < 1.0:
        counts_loss_weight = 1.0
        print("WARNING: you are training on low-read depth data")

    # adjust bias model for training  - using train and validation set
    # the bias model might be trained on a difference read depth compared to the given data - so this step scales the bias model to account for that
    #bias_model = param_utils.load_model_wrapper(args.bias_model_path)
    #bias_model_scaled = adjust_bias_model_logcounts(bias_model, nonpeak_seqs[(nonpeak_cnts< upper_thresh) & (nonpeak_cnts>lower_thresh)], nonpeak_cnts[(nonpeak_cnts< upper_thresh) & (nonpeak_cnts>lower_thresh)])
    # save the new bias model
    #bias_model_scaled.save("{}bias_model_scaled.h5".format(args.output_prefix))

    # store the parameters being used  - in a TSV file
    file = open("{}chrombpnet_data_params.tsv".format(args.output_prefix),"w")
    file.write("\t".join(["counts_sum_min_thresh", str(round(lower_thresh,2))]))
    file.write("\n")
    file.write("\t".join(["counts_sum_max_thresh", str(round(upper_thresh,2))]))
    file.write("\n")
    file.write("\t".join(["trainings_pts_post_thresh", str(sum((final_cnts<upper_thresh) & (final_cnts>lower_thresh)))]))
    file.write("\n")
    file.close()

    file = open("{}chrombpnet_model_params.tsv".format(args.output_prefix),"w")
    file.write("\t".join(["counts_loss_weight", str(round(counts_loss_weight,2))]))
    file.write("\n")
    file.write("\t".join(["filters", str(args.filters)]))
    file.write("\n")
    file.write("\t".join(["n_dil_layers", str(args.n_dilation_layers)]))
    file.write("\n")
    file.write("\t".join(["bias_model_path", "{}bias_model_scaled.h5".format(args.output_prefix)]))
    file.write("\n")
    file.write("\t".join(["inputlen", str(args.inputlen)]))
    file.write("\n")
    file.write("\t".join(["outputlen", str(args.outputlen)]))
    file.write("\n")
    file.write("\t".join(["max_jitter", str(args.max_jitter)]))
    file.write("\n")
    file.write("\t".join(["chr_fold_path", str(args.chr_fold_path)]))
    file.write("\n")
    file.write("\t".join(["negative_sampling_ratio", str(args.negative_sampling_ratio)]))
    file.close()

    return (args.bias_model_path, nonpeak_seqs[(nonpeak_cnts< upper_thresh) & (nonpeak_cnts>lower_thresh)], nonpeak_cnts[(nonpeak_cnts< upper_thresh) & (nonpeak_cnts>lower_thresh)], args.output_prefix)

In [ ]:
celltypes = ["alpha", "beta"]
states = ["ND", "T2D"]

# Create folders
for fold in range(5):
    # Make output directory
    output_directory = f"{chrombpnet_dir}/data/fold" + str(fold) + "/accessibility/"
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
        
# Define function to run a fold
def process_celltype_state_fold(celltype, state, fold):     
    # Define arguments
    args = argparse.Namespace(
        genome = genome_fasta_path,
        bigwig = f"{chrombpnet_dir}/data/{celltype}_{state}_coverage_unstranded.bw",
        peaks = f"{chrombpnet_dir}/data/peaks/{celltype}_correlated.bed",
        nonpeaks = f"{chrombpnet_dir}/data/fold{fold}/peaks/accessibility/{celltype}_negatives.bed",
        negative_sampling_ratio = 0.1,
        outlier_threshold = 0.9999,
        max_jitter = 500,
        chr_fold_path = f"{chrombpnet_dir}/data/fold_{fold}.json",
        inputlen = 2114,
        outputlen = 1000,
        filters = 512,
        n_dilation_layers = 8,
        bias_model_path = f"{chrombpnet_dir}/data/fold{fold}/bias/models/bias.h5",
        output_prefix = f"{chrombpnet_dir}/data/fold{fold}/accessibility/{celltype}_{state}_"
    )

    # Run the function
    result = get_hyper_params(args)
    return result

# Run celltypes and fold in parallel
res = []
for fold in range(5):
    for state in states:
        results = process_map(partial(process_celltype_state_fold, state = state, fold=fold), celltypes)
        res.append(results)

In [ ]:
from tqdm.notebook import tqdm
import chrombpnet.helpers.hyperparameters.param_utils as param_utils
# Adjust bias models (not in parallel due to GPU usage)
counter = 0
for fold in range(5):
    for state in states:
        results = res[counter]
        for i in tqdm(range(len(results))):
            bias_model = param_utils.load_model_wrapper(results[i][0])
            bias_model_scaled = adjust_bias_model_logcounts(bias_model, results[i][1], results[i][2])
            bias_model_scaled.save("{}bias_model_scaled.h5".format(results[i][3]))
        counter += 1

### Finetune accessibility models

In [ ]:
import chrombpnet.training.models.chrombpnet_with_bias_model as chrombpnet_with_bias_model
import chrombpnet.training.train as train
from chrombpnet.training.predict import load_model_wrapper
import os
import argparse
import pandas as pd
import chrombpnet.training.data_generators.initializers as initializers
import tensorflow.keras.callbacks as tfcallbacks
import chrombpnet.training.utils.losses as losses
import chrombpnet.training.utils.callbacks as callbacks
from __future__ import division, print_function, absolute_import
from tensorflow.keras.utils import get_custom_objects
import importlib.machinery
from chrombpnet.training.utils.losses import multinomial_nll
from tensorflow.keras.optimizers import Adam
import re

In [ ]:
def freeze_layer_by_name(model, target_name):
    for layer in model.layers:
        if layer.name == target_name:
            layer.trainable = False
            print(f"Froze layer: {layer.name}")
        # Recurse if nested
        if hasattr(layer, "layers"):
            freeze_layer_by_name(layer, target_name)

def set_layers_trainable_by_pattern(model, pattern, trainable=False):
    """
    Recursively freeze or unfreeze layers whose name matches the regex pattern.

    Args:
        model: Keras Model or Layer
        pattern: regex pattern to match layer names
        trainable: True to unfreeze, False to freeze
    """
    for layer in model.layers:
        if re.search(pattern, layer.name):
            layer.trainable = trainable
            action = "Unfroze" if trainable else "Froze"
            print(f"{action} layer: {layer.name}")
        # Recurse into nested layers
        if hasattr(layer, "layers"):
            set_layers_trainable_by_pattern(layer, pattern, trainable=trainable)

import tensorflow.keras as keras
class SafeLossHistory(keras.callbacks.Callback):
    """
    Callbacks to store train, validation loss at the the end of every batch and the end of every epoch.
    You can also track the counts loss and profile loss seperatley using the callbacks provided.
    """
    
    def __init__(self,model_output_path_logs_name,to_track):
        self.model_output_path_logs_name=model_output_path_logs_name
        self.to_track=to_track
        keras.callbacks.Callback.__init__(self)

    def on_train_begin(self, logs={}):
        self.outf=open(self.model_output_path_logs_name,'w')
        self.outf.write('Epoch\tBatch\t'+'\t'.join(self.to_track)+'\n')
        self.losses ={}

    def on_epoch_begin(self,epoch, logs={}):
        self.losses[epoch]={}
        for trackable in self.to_track:
            self.losses[epoch][trackable]=[]         
        self.cur_epoch=epoch
        
    def on_batch_end(self, batch, logs={}):
        for trackable in self.to_track:
            self.losses[self.cur_epoch][trackable].append(logs.get(trackable))
        
    def on_epoch_end(self,epoch,logs={}):
        marker=self.to_track[0] 
        num_batches=len(self.losses[self.cur_epoch][marker])
        for i in range(num_batches):
            self.outf.write(str(epoch)+'\t'+str(i))
            for trackable in self.to_track:
                self.outf.write('\t'+str(self.losses[epoch][trackable][i]))
            self.outf.write('\n')

        
    def on_train_end(self,logs={}):
        self.outf.close()            

In [ ]:
import tensorflow as tf

if tf.config.list_physical_devices('GPU'):
    print("GPU is available")
else:
    print("GPU not found")

In [ ]:
genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
celltypes = ["alpha", "beta"]
states = ["ND", "T2D"]
overwrite = False

# Fine-tune all models
for fold in range(5):
    for celltype in celltypes:
        for state in states:
            # Make output directory
            output_directory = f"{chrombpnet_dir}/data/fold" + str(fold) + "/accessibility/models/"
            if not os.path.exists(output_directory):
                os.makedirs(output_directory)
            
            # Check if the model has already been run
            model_check_path = F"{output_directory}{celltype}_{state}_nobias.h5"
            if not os.path.exists(model_check_path) or overwrite:
                # Set count loss weight to 5
                param_file = f"{chrombpnet_dir}/data/fold" + str(fold) + "/accessibility/" + str(celltype) + "_chrombpnet_model_params.tsv"
                params = pd.read_csv(param_file, sep="\t", header=None)
                params.iloc[0,1] = str(5)
                params.to_csv(f"{chrombpnet_dir}/test_params.tsv", sep="\t", header=None, index=None)
                
                # Define arguments
                args = argparse.Namespace(
                    genome = genome_fasta_path,
                    bigwig = f"{chrombpnet_dir}/data/{celltype}_{state}_coverage_unstranded.bw",
                    peaks = f"{chrombpnet_dir}/data/fold{fold}/accessibility/{celltype}_{state}_filtered.peaks.bed",
                    nonpeaks = f"{chrombpnet_dir}/data/fold{fold}/accessibility/{celltype}_{state}_filtered.nonpeaks.bed",
                    params = f"{chrombpnet_dir}/test_params.tsv",
                    output_prefix = f"{output_directory}{celltype}_{state}",
                    chr_fold_path = f"{chrombpnet_dir}/data/fold_" + str(fold) + ".json",
                    epochs = 30,
                    early_stop = 5,
                    batch_size = 64,
                    learning_rate = 0.0001,
                    trackables =  ['loss','val_loss'],
                    seed = 1234,
                    architecture_from_file = chrombpnet_with_bias_model.__file__
                )   
                
                # Initialize generators to load data
                parameters = train.get_model_param_dict(args)
                np.random.seed(args.seed)
                train_generator = initializers.initialize_generators(args, "train", parameters, return_coords=False)
                valid_generator = initializers.initialize_generators(args, "valid", parameters, return_coords=False)
            
                # Setup
                model_output_path_h5_name=args.output_prefix+".h5"
                model_output_path_logs_name=args.output_prefix+".log"
                checkpointer = tfcallbacks.ModelCheckpoint(filepath=model_output_path_h5_name, monitor="val_loss", mode="min",  verbose=1, save_best_only=True)
                earlystopper = tfcallbacks.EarlyStopping(monitor='val_loss', mode="min", patience=args.early_stop, verbose=1, restore_best_weights=True)
                history= SafeLossHistory(model_output_path_logs_name+".batch",args.trackables)
                csvlogger = tfcallbacks.CSVLogger(model_output_path_logs_name, append=False)
                cur_callbacks=[checkpointer,earlystopper,csvlogger,history]
            
                # Load base model
                architecture_module=importlib.machinery.SourceFileLoader('',args.architecture_from_file).load_module()
                model=architecture_module.getModelGivenModelOptionsAndWeightInits(args, parameters)
                model.load_weights(f"{output_directory}{celltype}.h5")
                
                # Freeze first convolutional layer
                freeze_layer_by_name(model, "wo_bias_bpnet_1st_conv")
            
                # Freeze convolutional stack
                set_layers_trainable_by_pattern(model, r"wo_bias_bpnet_\d+conv", trainable=False)
            
                # Compile the model
                counts_loss_weight=float(parameters['counts_loss_weight'])
                model.compile(optimizer=Adam(learning_rate=args.learning_rate), loss=[multinomial_nll,'mse'], loss_weights=[1,counts_loss_weight])
            
                # Fit the model
                model.fit(train_generator, validation_data=valid_generator, epochs=args.epochs, verbose=1, callbacks=cur_callbacks)
            
                # Unfreeze the convolutional stack
                set_layers_trainable_by_pattern(model, r"wo_bias_bpnet_\d+conv", trainable=True)
            
                # Compile the model
                counts_loss_weight=float(parameters['counts_loss_weight'])
                model.compile(optimizer=Adam(learning_rate=args.learning_rate), loss=[multinomial_nll,'mse'], loss_weights=[1,counts_loss_weight])
            
                # Fresh checkpointer, stopper, history and csvlogger
                checkpointer = tfcallbacks.ModelCheckpoint(filepath=model_output_path_h5_name, monitor="val_loss", mode="min",  verbose=1, save_best_only=True)
                earlystopper = tfcallbacks.EarlyStopping(monitor='val_loss', mode="min", patience=args.early_stop, verbose=1, restore_best_weights=True)
                history= SafeLossHistory(model_output_path_logs_name+".batch",args.trackables)
                csvlogger = tfcallbacks.CSVLogger(model_output_path_logs_name, append=False)
                cur_callbacks=[checkpointer,earlystopper,csvlogger,history]
                
                # Fit the model
                model.fit(train_generator, validation_data=valid_generator, epochs=args.epochs, verbose=1, callbacks=cur_callbacks)
            
                # Save the model
                model.save(model_output_path_h5_name)
                architecture_module.save_model_without_bias(model, args.output_prefix)

### Predict

In [ ]:
import chrombpnet.training.train as load_model_wrapper
import argparse
import pandas as pd
import os
import scipy.stats
import numpy as np
import json
import h5py
import tensorflow as tf
import chrombpnet.training.utils.argmanager as argmanager
import chrombpnet.training.utils.losses as losses
import chrombpnet.training.metrics as metrics
import chrombpnet.training.data_generators.initializers as initializers
from tensorflow.keras.utils import get_custom_objects
from tensorflow.keras.models import load_model
import chrombpnet.training.data_generators.batchgen_generator as batchgen_generator
from chrombpnet.training.utils import data_utils
from chrombpnet.training.predict import predict_on_batch_wrapper, write_predictions_h5py
import seaborn as sns
from scipy.stats import spearmanr, pearsonr
from chrombpnet.training.models.chrombpnet_with_bias_model import load_pretrained_bias, bpnet_model
from tensorflow.keras.backend import int_shape
from tensorflow.keras.layers import Input, Cropping1D, add, Conv1D, GlobalAvgPool1D, Dense, Add, Concatenate, Lambda, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Model
import random as rn

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
%matplotlib inline

In [ ]:
# Variables
celltypes = ["alpha", "beta"]
states = ["ND", "T2D"]
modeltypes = ["ND", "T2D","Basal"]

# Setup dict
res_dict = {}
for celltype in celltypes:
    res_dict[celltype] = {}
    for state in states:
        res_dict[celltype][state] = {}
        for modeltype in modeltypes:
            res_dict[celltype][state][modeltype] = {}
            for mode in ['train', 'valid', 'test']:
                res_dict[celltype][state][modeltype][mode] = {}
                for fold in range(5):
                    res_dict[celltype][state][modeltype][mode][fold] = {}


In [ ]:
counter = 0
for celltype in celltypes:
    for state in states:
        for modeltype in modeltypes:
            for fold in range(5):
                for mode in ['train', 'valid', 'test']:
                    if modeltype == "Basal":
                        model_path = f"{chrombpnet_dir}/data/fold{fold}/accessibility/models/{celltype}.h5"
                    else:
                        model_path = f"{chrombpnet_dir}/data/fold{fold}/accessibility/models/{celltype}_{modeltype}.h5"
                    if os.path.exists(model_path):
                        chr_fold_path = f"{chrombpnet_dir}/data/fold_{fold}.json"
                        peaks_path = f"{chrombpnet_dir}/data/peaks/{celltype}_correlated.bed"
                        genome_fasta_path = f"{chrombpnet_dir}/data/hg38.fa"
                        batch_size = 64
                        bigwig_path = f"{chrombpnet_dir}/data/{celltype}_{state}_coverage_unstranded.bw"
                        output_file = f"{chrombpnet_dir}/data/fold{fold}/accessibility/predictions/{celltype}_{state}_{modeltype}_{mode}.h5"

                        # Define the model object
                        #custom_objects={"tf": tf, "multinomial_nll":losses.multinomial_nll, "logcount_predictions": logcount_predictions}    
                        custom_objects={"tf": tf, "multinomial_nll":losses.multinomial_nll}
                        get_custom_objects().update(custom_objects)    
                        model=load_model(model_path, compile=False)
                        
                        # Create data generator
                        splits_dict=json.load(open(chr_fold_path))
                        
                        # Peaks
                        NARROWPEAK_SCHEMA = ["chr", "start", "end", "1", "2", "3", "4", "5", "6", "summit"]
                        peak_regions=pd.read_csv(peaks_path,header=None,sep='\t',names=NARROWPEAK_SCHEMA)
                        chroms=splits_dict[mode]
                        peak_regions=peak_regions[peak_regions["chr"].isin(chroms)]
                        
                        # Static params
                        inputlen = 2114
                        outputlen = 1000
                        negative_sampling_ratio = 1.0
                        max_jitter = 0
                        add_revcomp = False
                        shuffle_at_epoch_start = False
                        nonpeak_regions = None
                        
                        # Create the generator
                        test_generator=batchgen_generator.ChromBPNetBatchGenerator(
                                                            peak_regions=peak_regions,
                                                            nonpeak_regions=nonpeak_regions,
                                                            genome_fasta=genome_fasta_path,
                                                            batch_size=batch_size,
                                                            inputlen=inputlen,                                        
                                                            outputlen=outputlen,
                                                            max_jitter=0,
                                                            negative_sampling_ratio=0,
                                                            cts_bw_file=bigwig_path,
                                                            add_revcomp=False,
                                                            return_coords=True,
                                                            shuffle_at_epoch_start=shuffle_at_epoch_start
                                                            )
                        
                        # Predictions
                        true_counts, profile_probs_predictions, true_counts_sum, counts_sum_predictions, coordinates = predict_on_batch_wrapper(model, test_generator)
                        
                        # Metrics
                        mnll_pw, mnll_norm, jsd_pw, jsd_norm, jsd_rnd, jsd_rnd_norm, mnll_rnd, mnll_rnd_norm =  metrics.profile_metrics(true_counts,profile_probs_predictions)
                        spearman_cor=spearmanr(true_counts_sum,counts_sum_predictions)[0]
                        pearson_cor=pearsonr(true_counts_sum,counts_sum_predictions)[0]  
                        mse=((true_counts_sum - counts_sum_predictions)**2).mean(axis=0)

                        # Collect summaries in a pandas dataframe
                        res_tmp = pd.DataFrame({
                            'Celltype': celltype,
                            'State': state,
                            'Model': modeltype,
                            'Fold': fold,
                            'Type': mode,
                            'MSE': mse,
                            'Pearson': pearson_cor,
                            'Spearman': spearman_cor,
                            'JSD_pw': np.nanmedian(jsd_pw),
                            'JSD_norm': np.nanmedian(jsd_norm),
                            'JSD_rand_pw': np.nanmedian(jsd_rnd),
                            'JSD_rand_norm': np.nanmedian(jsd_rnd_norm)
                        }, index = [counter])
                
                        # Combine
                        if counter == 0:
                            met = res_tmp
                        else:
                            met = pd.concat((met, res_tmp))

                        # Collect raws in dictionaries
                        res_dict[celltype][state][modeltype][mode][fold] = {'JSD': jsd_pw, 'JSD_rand': jsd_rnd, 'Truth': true_counts_sum, 'Pred': counts_sum_predictions}
                        
                        # Write to h5 file
                        write_predictions_h5py(output_file, profile_probs_predictions, counts_sum_predictions, coordinates)
                        met.to_csv(f"{chrombpnet_dir}/finetune_metrics.csv")
                
                        # Counter
                        counter += 1

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(7,5))

# Select mode
mode = "valid"
celltype = "beta"
for stateidx in range(len(states)):
    modeltype = modeltypes[stateidx] 
    state = states[stateidx] 
    
    # Extract across folds
    x, y, jsd, jsd_rand = [], [], [], []
    for fold in range(5):
        x.append(res_dict[celltype][state][modeltype][mode][fold]['Truth'])
        y.append(res_dict[celltype][state][modeltype][mode][fold]['Pred'])
        jsd.append(res_dict[celltype][state][modeltype][mode][fold]['JSD'])
        jsd_rand.append(res_dict[celltype][state][modeltype][mode][fold]['JSD_rand'])
    
    # Stack
    x = np.hstack(x)
    y = np.hstack(y)
    jsd = np.hstack(jsd)
    jsd_rand = np.hstack(jsd_rand)
    
    # Calculate MSE and Spearman
    mse = ((x - y)**2).mean(axis=0)
    spearman_corr, _ = spearmanr(x, y)
    
    # Define fig
    hb = ax[0,stateidx].hexbin(x, y, gridsize=40, mincnt=1)  # store hexbin object
    
    # add colorbar with counts
    cb = fig.colorbar(hb, ax=ax[0,stateidx])
    cb.set_label("Counts per bin")
    
    #ax[stateidx,typeidx].set_xlabel("True counts sum")
    #ax[stateidx,typeidx].set_ylabel("Predicted counts sum")
    ax[0,stateidx].set_title(f"State: {state}, Model: {modeltype}")
    
    # annotation
    ax[0,stateidx].text(
        0.05, 0.95,
        f"MSE = {mse:.3f}",
        transform=ax[0,stateidx].transAxes,
        verticalalignment="top",
        bbox=dict(boxstyle="round", alpha=0.3)
    )
    
    ax[0,stateidx].legend()

    # JSD
    sns.violinplot(ax = ax[1,stateidx], x = 'Group', y = 'JSD', data=pd.concat([pd.DataFrame({'JSD': jsd, 'Group': 'Sites'}), pd.DataFrame({'JSD': jsd_rand, 'Group': 'Randomized'})]))
    ax[1,stateidx].set_ylim(0,1)

plt.savefig(f"{chrombpnet_dir}/beta_model_evaluation.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(7,5))

# Select mode
mode = "valid"
celltype = "alpha"
for stateidx in range(len(states)):
    modeltype = modeltypes[stateidx] 
    state = states[stateidx] 
    
    # Extract across folds
    x, y, jsd, jsd_rand = [], [], [], []
    for fold in range(5):
        x.append(res_dict[celltype][state][modeltype][mode][fold]['Truth'])
        y.append(res_dict[celltype][state][modeltype][mode][fold]['Pred'])
        jsd.append(res_dict[celltype][state][modeltype][mode][fold]['JSD'])
        jsd_rand.append(res_dict[celltype][state][modeltype][mode][fold]['JSD_rand'])
    
    # Stack
    x = np.hstack(x)
    y = np.hstack(y)
    jsd = np.hstack(jsd)
    jsd_rand = np.hstack(jsd_rand)
    
    # Calculate MSE and Spearman
    mse = ((x - y)**2).mean(axis=0)
    spearman_corr, _ = spearmanr(x, y)
    
    # Define fig
    hb = ax[0,stateidx].hexbin(x, y, gridsize=40, mincnt=1)  # store hexbin object
    
    # add colorbar with counts
    cb = fig.colorbar(hb, ax=ax[0,stateidx])
    cb.set_label("Counts per bin")
    
    #ax[stateidx,typeidx].set_xlabel("True counts sum")
    #ax[stateidx,typeidx].set_ylabel("Predicted counts sum")
    ax[0,stateidx].set_title(f"State: {state}, Model: {modeltype}")
    
    # annotation
    ax[0,stateidx].text(
        0.05, 0.95,
        f"MSE = {mse:.3f}",
        transform=ax[0,stateidx].transAxes,
        verticalalignment="top",
        bbox=dict(boxstyle="round", alpha=0.3)
    )
    
    ax[0,stateidx].legend()

    # JSD
    sns.violinplot(ax = ax[1,stateidx], x = 'Group', y = 'JSD', data=pd.concat([pd.DataFrame({'JSD': jsd, 'Group': 'Sites'}), pd.DataFrame({'JSD': jsd_rand, 'Group': 'Randomized'})]))
    ax[1,stateidx].set_ylim(0,1)

plt.savefig(f"{chrombpnet_dir}/alpha_model_evaluation.pdf", format="pdf", bbox_inches="tight")
plt.show()

In [ ]:
met["Group"] = met["State"] + "_" + met["Model"]

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(7,5))

data = met[met["Celltype"] == "beta"]

sns.barplot(x="Type", y="MSE", hue="Group", data=data, ax=ax[0,0])
sns.barplot(x="Type", y="Pearson", hue="Group", data=data, ax=ax[0,1])
ax[0,1].set_ylim(0,1)

sns.barplot(x="Type", y="JSD_pw", hue="Group", data=data, ax=ax[0,2])
ax[0,2].set_ylim(0,1)

data = met[met["Celltype"] == "alpha"]

sns.barplot(x="Type", y="MSE", hue="Group", data=data, ax=ax[1,0])
sns.barplot(x="Type", y="Pearson", hue="Group", data=data, ax=ax[1,1])
ax[1,1].set_ylim(0,1)

sns.barplot(x="Type", y="JSD_pw", hue="Group", data=data, ax=ax[1,2])
ax[1,2].set_ylim(0,1)

# Get legend handles from first axis
handles, labels = ax[0,0].get_legend_handles_labels()

# Remove legends from all axes
for a in ax.flat:
    if a.get_legend() is not None:
        a.get_legend().remove()

# Add one global legend outside the plot
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(0.98, 0.5), frameon=False)

# leave room on the right for legend (adjust 0.86 -> smaller to leave more space)
fig.tight_layout(rect=[0, 0, 0.98, 1])
plt.savefig(f"{chrombpnet_dir}/finetune_eval.pdf", format="pdf", bbox_inches="tight")
plt.show()